# IPL Analytics — Day 31 Merged Dashboard Repair

This merged notebook combines `31_dashboard_live_issues_repair` and `31_dashboard_prediction_csrf_DL_FINAL` in one ordered workflow. It includes the autocomplete/dropdown, removal of unsupported Opposition and Recent form options, CSRF-safe ML POST handling, DL sequence endpoint/preprocessing wiring, route registration, backups, and final configuration checks.

**Run from the first cell to the last cell in your repository's Conda environment.** Review the printed backup paths and acceptance checks. `manage.py check` does not prove the ML/DL models run end-to-end; complete the browser acceptance tests at the end.


In [69]:
from pathlib import Path
import datetime, shutil, subprocess, sys, re, json, csv, hashlib

CWD = Path.cwd().resolve()
ROOT = next((p for p in [CWD, *CWD.parents]
             if (p / "django_api" / "manage.py").exists()
             and (p / "django_api" / "ipl_api").exists()), None)
if ROOT is None:
    raise FileNotFoundError(r"Set the notebook working directory to D:\Git\Personal\ipl-player-performance")
DJANGO_DIR = ROOT / "django_api"
APP_DIR = DJANGO_DIR / "ipl_api"
DOC_DIR = ROOT / "docs" / "dashboard_live_repair"
BACKUP_DIR = ROOT / "backups" / ("dashboard_live_repair_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
DOC_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Django app:", APP_DIR)
print("Backup directory:", BACKUP_DIR)


Repository: D:\Git\Personal\ipl-player-performance
Django app: D:\Git\Personal\ipl-player-performance\django_api\ipl_api
Backup directory: D:\Git\Personal\ipl-player-performance\backups\dashboard_live_repair_20261024_020133


## 1. Inspect the actual API, dashboard assets, and route configuration

In [70]:
files = [
    APP_DIR / "urls.py",
    APP_DIR / "views.py",
    APP_DIR / "eda_views.py",
    APP_DIR / "prediction_views.py",
    APP_DIR / "player_name_prediction_views_day28.py",
    APP_DIR / "dashboard_views_day26.py",
]
for p in files:
    print("\\n---", p.relative_to(ROOT), "exists:", p.exists(), "---")
    if p.exists():
        print(p.read_text(encoding="utf-8")[:10000])

for folder in [APP_DIR / "templates" / "ipl_api", APP_DIR / "static" / "ipl_api" / "js"]:
    if folder.exists():
        for p in sorted(folder.glob("*")):
            if p.is_file() and any(x in p.name.lower() for x in ["dashboard", "prediction", "player_name"]):
                print("ASSET:", p.relative_to(ROOT))


\n--- django_api\ipl_api\urls.py exists: True ---
from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView
from .dashboard_views_day26 import dashboard_day25
from .eda_summary_day26 import eda_summary_day26
from .eda_views import EDABattingAPIView, EDABowlingAPIView, EDASeasonAPIView, EDAVenueAPIView, EDAOppositionAPIView, EDARecentFormAPIView
from .prediction_views import BattingPredictionAPIView, BowlingPredictionAPIView, UnifiedPredictionAPIView
from .batch_prediction_views import BatchPredictionAPIView
from .prediction_center_views_day28 import prediction_center_day28
from .player_name_prediction_views_day28 import player_name_prediction_day28, player_name_options_day28, model_metrics_day28

urlpatterns = [
    path("dashboard/", dashboard_day25, name="ipl-dashboard"),
    path("predictions/", prediction_center_day28, name="ipl-predictions"),
    path("players/", PlayerListAPIView.as_view(), name="player-list"),
    path("players/search-options/", play

## 2. Back up relevant source files before any edits

In [71]:
targets = []
for p in files:
    if p.exists():
        targets.append(p)
for folder in [APP_DIR / "templates" / "ipl_api", APP_DIR / "static" / "ipl_api" / "js", APP_DIR / "static" / "ipl_api" / "css"]:
    if folder.exists():
        targets.extend(p for p in folder.iterdir() if p.is_file() and any(x in p.name.lower() for x in ["dashboard", "prediction", "player_name"]))
for p in dict.fromkeys(targets):
    dest = BACKUP_DIR / p.relative_to(ROOT)
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dest)
    print("Backup:", dest.relative_to(ROOT))


Backup: backups\dashboard_live_repair_20261024_020133\django_api\ipl_api\urls.py
Backup: backups\dashboard_live_repair_20261024_020133\django_api\ipl_api\views.py
Backup: backups\dashboard_live_repair_20261024_020133\django_api\ipl_api\eda_views.py
Backup: backups\dashboard_live_repair_20261024_020133\django_api\ipl_api\prediction_views.py
Backup: backups\dashboard_live_repair_20261024_020133\django_api\ipl_api\player_name_prediction_views_day28.py
Backup: backups\dashboard_live_repair_20261024_020133\django_api\ipl_api\dashboard_views_day26.py
Backup: backups\dashboard_live_repair_20261024_020133\django_api\ipl_api\templates\ipl_api\dashboard_day25.html
Backup: backups\dashboard_live_repair_20261024_020133\django_api\ipl_api\templates\ipl_api\dashboard_day25.html.day25_backup
Backup: backups\dashboard_live_repair_20261024_020133\django_api\ipl_api\templates\ipl_api\dashboard_day25.html.day27_backup_20261023_160025
Backup: backups\dashboard_live_repair_20261024_020133\django_api\ipl_ap

## 3. Add a real health endpoint if it is not already registered

In [72]:
urls_path = APP_DIR / "urls.py"
health_module = APP_DIR / "health_views_repair.py"

health_source = """from django.db import connection
from django.http import JsonResponse

def health_status(request):
    try:
        with connection.cursor() as cursor:
            cursor.execute("SELECT 1")
            cursor.fetchone()
        return JsonResponse({"status": "ok", "database": "ok"}, status=200)
    except Exception:
        return JsonResponse({"status": "degraded", "database": "error"}, status=503)
"""
health_module.write_text(health_source, encoding="utf-8")

if not urls_path.exists():
    raise FileNotFoundError(urls_path)

urls = urls_path.read_text(encoding="utf-8")
if 'path("health/"' in urls or "path('health/'" in urls:
    print("A health route already exists; no duplicate route added.")
else:
    if "from .health_views_repair import health_status" not in urls:
        urls = "from .health_views_repair import health_status\n" + urls
    marker = "urlpatterns = ["
    pos = urls.find(marker)
    if pos < 0:
        raise RuntimeError("Could not find 'urlpatterns = ['; URL file was not modified.")
    insert_at = pos + len(marker)
    urls = urls[:insert_at] + '\n    path("health/", health_status, name="health-status"),' + urls[insert_at:]
    urls_path.write_text(urls, encoding="utf-8")
    print("Registered /api/health/ route.")

# Compile the two files before proceeding.
compile(health_module.read_text(encoding="utf-8"), str(health_module), "exec")
compile(urls_path.read_text(encoding="utf-8"), str(urls_path), "exec")
print("Health view and URL configuration syntax checks passed.")


Registered /api/health/ route.
Health view and URL configuration syntax checks passed.


## 4. Audit database columns used by EDA

In [73]:
schema_code = r'''
from django.db import connection
import json
tables = ["stg_ipl_batting_player_match", "stg_ipl_bowling_player_match", "stg_ipl_match_metadata"]
result = {}
with connection.cursor() as cursor:
    for table in tables:
        try:
            cursor.execute("SHOW COLUMNS FROM `" + table + "`")
            result[table] = [row[0] for row in cursor.fetchall()]
        except Exception as exc:
            result[table] = {"error": str(exc)}
print(json.dumps(result, indent=2))
'''
proc = subprocess.run([sys.executable, "manage.py", "shell", "-c", schema_code],
                      cwd=DJANGO_DIR, capture_output=True, text=True)
print(proc.stdout)
if proc.stderr:
    print("STDERR:", proc.stderr[-2500:])
print("Schema audit exit code:", proc.returncode)
print("Use this output to wire Opposition EDA to a real column; do not assume metadata contains opposition.")


12 objects imported automatically (use -v 2 for details).

{
  "stg_ipl_batting_player_match": [
    "match_id",
    "date",
    "season",
    "venue",
    "city",
    "team_1",
    "team_2",
    "match_type",
    "team",
    "opposition",
    "player_name",
    "player_id",
    "player_name_norm",
    "team_norm",
    "opposition_norm",
    "runs",
    "balls_faced",
    "fours",
    "sixes",
    "batting_position",
    "was_dismissed",
    "dismissal_kind",
    "innings_count",
    "strike_rate"
  ],
  "stg_ipl_bowling_player_match": [
    "match_id",
    "date",
    "season",
    "venue",
    "city",
    "team_1",
    "team_2",
    "match_type",
    "team",
    "opposition",
    "player_name",
    "player_id",
    "player_name_norm",
    "team_norm",
    "opposition_norm",
    "legal_balls",
    "runs_conceded",
    "wickets",
    "wides",
    "noballs",
    "dots",
    "innings_count",
    "overs",
    "economy"
  ],
  "stg_ipl_match_metadata": [
    "match_id",
    "source_file",


## 5. Inspect the ML 403 cause — do not disable CSRF globally

In [74]:
for p in (APP_DIR / "static" / "ipl_api" / "js").glob("*.js"):
    if any(x in p.name.lower() for x in ["dashboard", "prediction", "player_name"]):
        lines = p.read_text(encoding="utf-8").splitlines()
        matches = [(i, line) for i, line in enumerate(lines, 1)
                   if any(x in line.lower() for x in ["fetch(", "csrf", "x-csrftoken", "predict/player", "credentials"])]
        if matches:
            print("\\n---", p.relative_to(ROOT), "---")
            for i, line in matches:
                print(f"{i}: {line[:250]}")
pv = APP_DIR / "player_name_prediction_views_day28.py"
if pv.exists():
    print("\\n--- player-name view ---\\n", pv.read_text(encoding="utf-8")[:10000])
print("\\nLikely fix: send the csrftoken cookie in X-CSRFToken for same-origin POST requests if SessionAuthentication is used. Confirm the view's permission/authentication classes first.")


\n--- django_api\ipl_api\static\ipl_api\js\dashboard_day25.js ---
11:     const response = await fetch(url, {
12:       credentials: "same-origin",
88:         show(resultId, await request("/api/predict/player/", {method:"POST", body:JSON.stringify(payload)}));
\n--- django_api\ipl_api\static\ipl_api\js\dashboard_day28.js ---
5:   const response = await fetch(url, {headers: {"Accept":"application/json", ...(options.headers || {})}, ...options});
\n--- django_api\ipl_api\static\ipl_api\js\dashboard_requested_features.js ---
2: (function day31InstallCsrfFetch() {
3:   if (window.__day31CsrfFetchInstalled) return;
4:   window.__day31CsrfFetchInstalled = true;
11:       const match = document.cookie.match(/(?:^|; )csrftoken=([^;]+)/);
14:       if (token && !headers.has("X-CSRFToken")) headers.set("X-CSRFToken", token);
15:       init = Object.assign({}, init, {headers: headers, credentials: init.credentials || "same-origin"});
17:     return originalFetch(input, init);
31:     const respo

## 6. Resolve the player partial-name issue

In [75]:
# Remove unsupported Opposition and Recent form choices from dashboard UI assets.
# Only edit dashboard-related HTML/JS files, and only after the backup cell has run.
import re

asset_files = []
for folder in [APP_DIR / "templates" / "ipl_api", APP_DIR / "static" / "ipl_api" / "js"]:
    if folder.exists():
        for p in folder.rglob("*"):
            if p.is_file() and p.suffix.lower() in {".html", ".js"} and any(
                token in p.name.lower() or token in p.read_text(encoding="utf-8", errors="ignore").lower()
                for token in ["dashboard", "opposition", "recent-form", "recent form"]
            ):
                asset_files.append(p)

def remove_option_blocks(text, terms):
    # Remove <option ...>Label/slug...</option> for unsupported EDA options.
    for term in terms:
        text = re.sub(
            r"<option\b(?:(?!</option>).)*?(?:" + re.escape(term) + r")(?:(?!</option>).)*?</option>",
            "", text, flags=re.I | re.S
        )
    return text

changed = []
for p in sorted(set(asset_files)):
    old = p.read_text(encoding="utf-8", errors="ignore")
    new = old
    new = remove_option_blocks(new, ["opposition", "recent.form", "recent-form", "recent_form"])
    # Remove common JS option-list entries and EDA map keys only when they clearly name these options.
    new = re.sub(r'^\s*\{\s*(?:label\s*:\s*)?["\']?Opposition["\']?\s*,.*?\},?\s*$', '', new, flags=re.I | re.M)
    new = re.sub(r'^\s*\{\s*(?:label\s*:\s*)?["\']?Recent form["\']?\s*,.*?\},?\s*$', '', new, flags=re.I | re.M)
    # Remove string entries from option arrays; do not delete endpoint implementations.
    new = re.sub(r'(["\'](?:opposition|recent-form|recent_form)["\']\s*:\s*["\'][^"\']*["\']\s*,?)', '', new, flags=re.I)
    if new != old:
        p.write_text(new, encoding="utf-8")
        changed.append(str(p.relative_to(ROOT)))
print("Updated dashboard assets:", changed if changed else "No matching option blocks found.")
print("If the choices remain, inspect the printed dashboard assets in Section 1; their markup may use a different structure.")


Updated dashboard assets: ['django_api\\ipl_api\\templates\\ipl_api\\dashboard_day25.html']
If the choices remain, inspect the printed dashboard assets in Section 1; their markup may use a different structure.


## 6A. Player search and prediction POST diagnostics

The dashboard must first call `GET /api/players/?search=<partial text>` and use a returned canonical `player_name` for detail requests. A partial string such as `raina` should not be sent directly to `/api/players/raina/` if that route performs exact matching. The next diagnostic cell prints the relevant JS request lines and endpoint configuration. For HTTP 403, inspect the response body and Django REST Framework authentication/permission settings; do not disable CSRF globally.

In [76]:
# Show exact frontend requests and backend permission/authentication declarations for a targeted fix.
for folder in [APP_DIR / "static" / "ipl_api" / "js", APP_DIR]:
    if folder.exists():
        for p in folder.rglob("*.js") if folder.name == "js" else folder.glob("*prediction*.py"):
            if p.is_file():
                text = p.read_text(encoding="utf-8", errors="ignore")
                matches = [(i, line.strip()) for i, line in enumerate(text.splitlines(), 1)
                           if any(k in line.lower() for k in [
                               "/api/players/", "/api/predict/player/", "x-csrftoken",
                               "csrf", "permission_classes", "authentication_classes",
                               "sessionauthentication", "allowany", "fetch("
                           ])]
                if matches:
                    print("\n---", p.relative_to(ROOT), "---")
                    for line_no, line in matches[:80]:
                        print(f"{line_no}: {line[:300]}")
print("\nDo not treat HTTP 403 as fixed until the browser request succeeds with the intended auth/CSRF policy.")



--- django_api\ipl_api\static\ipl_api\js\dashboard_day25.js ---
11: const response = await fetch(url, {
61: try { show("playerResult", await request("/api/players/" + encodeURIComponent(name) + "/")); }
68: const data = await request("/api/players/search-options/");
88: show(resultId, await request("/api/predict/player/", {method:"POST", body:JSON.stringify(payload)}));

--- django_api\ipl_api\static\ipl_api\js\dashboard_day28.js ---
5: const response = await fetch(url, {headers: {"Accept":"application/json", ...(options.headers || {})}, ...options});

--- django_api\ipl_api\static\ipl_api\js\dashboard_requested_features.js ---
2: (function day31InstallCsrfFetch() {
3: if (window.__day31CsrfFetchInstalled) return;
4: window.__day31CsrfFetchInstalled = true;
11: const match = document.cookie.match(/(?:^|; )csrftoken=([^;]+)/);
14: if (token && !headers.has("X-CSRFToken")) headers.set("X-CSRFToken", token);
17: return originalFetch(input, init);
31: const response = await fetch(url, {
8

## 6B. Player-name dropdowns and DL endpoint registration

Adds autocomplete for player-name inputs and a dedicated DL sequence endpoint. The DL endpoint validates a 5-match numeric sequence and loads only a saved model found in the Day 12 manifest. The input must already use the saved training-time preprocessing; the endpoint returns a validation error rather than inventing a prediction.


In [77]:

from pathlib import Path
import re

# Suggestions API: partial-name lookup against the project's MySQL staging tables.
suggestion_path = APP_DIR / "player_suggestions_day31.py"
suggestion_path.write_text('''from django.db import connection
from django.http import JsonResponse
from django.views.decorators.http import require_GET

@require_GET
def player_suggestions(request):
    q = (request.GET.get("q") or "").strip()
    if not q:
        return JsonResponse({"results": []})
    try:
        with connection.cursor() as cursor:
            cursor.execute(
                "SELECT player_name FROM ("
                "SELECT player_name FROM stg_ipl_batting_player_match WHERE player_name IS NOT NULL AND TRIM(player_name) <> '' "
                "UNION SELECT player_name FROM stg_ipl_bowling_player_match WHERE player_name IS NOT NULL AND TRIM(player_name) <> ''"
                ") p WHERE LOWER(player_name) LIKE LOWER(%s) GROUP BY player_name ORDER BY player_name LIMIT 20",
                [f"%{q}%"])
            rows = [{"player_name": r[0]} for r in cursor.fetchall()]
        return JsonResponse({"results": rows})
    except Exception as exc:
        return JsonResponse({"detail": "Suggestions unavailable", "error": str(exc)}, status=503)
''', encoding="utf-8")

# DL endpoint: accepts exact 5-row sequence; only loads saved model if manifest and target match.
dl_path = APP_DIR / "deep_learning_prediction_day31.py"
dl_path.write_text('''import json
from pathlib import Path
import numpy as np
from django.conf import settings
from django.http import JsonResponse
from django.views.decorators.http import require_POST
from django.views.decorators.csrf import csrf_exempt

@csrf_exempt
@require_POST
def deep_learning_prediction(request):
    try:
        payload = json.loads(request.body.decode("utf-8"))
    except Exception:
        return JsonResponse({"detail": "Body must be valid JSON."}, status=400)
    target = payload.get("target")
    if target not in {"next_match_runs", "next_match_wickets", "next_match_runs_conceded"}:
        return JsonResponse({"detail": "Unsupported target."}, status=400)
    try:
        arr = np.asarray(payload.get("sequence"), dtype=np.float32)
    except (TypeError, ValueError):
        return JsonResponse({"detail": "sequence must be numeric."}, status=400)
    if arr.ndim != 2 or arr.shape[0] != 5 or arr.shape[1] < 1 or not np.isfinite(arr).all():
        return JsonResponse({"detail": "sequence must be a finite numeric 5 x feature_count array."}, status=400)
    if payload.get("preprocessed") is not True:
        return JsonResponse({"detail": "Apply the saved Day 12 imputer/scaler before submitting; raw inputs are rejected."}, status=400)
    root = Path(settings.BASE_DIR).parent
    manifests = [root / "reports" / "day12" / "model_artifacts_manifest.csv",
                 root / "reports" / "day12_model_artifacts_manifest.csv"]
    manifest = next((p for p in manifests if p.exists()), None)
    if manifest is None:
        return JsonResponse({"detail": "Day 12 model_artifacts_manifest.csv not found."}, status=503)
    try:
        import pandas as pd
        df = pd.read_csv(manifest)
        tc = next((c for c in ["target", "target_name", "prediction_target"] if c in df.columns), None)
        mc = next((c for c in ["model_path", "model_file", "artifact_path"] if c in df.columns), None)
        if not tc or not mc:
            return JsonResponse({"detail": "Manifest requires target and model_path columns."}, status=503)
        found = df[df[tc].astype(str).eq(target)]
        if found.empty:
            return JsonResponse({"detail": "No saved DL model registered for target."}, status=503)
        model_path = Path(str(found.iloc[0][mc]))
        if not model_path.is_absolute():
            model_path = root / model_path
        if not model_path.exists():
            parts = list(model_path.parts)
            if "models" in parts:
                model_path = root.joinpath(*parts[parts.index("models"):])
        if not model_path.exists():
            return JsonResponse({"detail": "Saved model file not found.", "path": str(model_path)}, status=503)
        import tensorflow as tf
        model = tf.keras.models.load_model(model_path)
        expected = tuple(model.input_shape[1:])
        if len(expected) != 2 or expected[0] not in (None, 5) or expected[1] not in (None, arr.shape[1]):
            return JsonResponse({"detail": "Input shape does not match saved model.", "expected": list(model.input_shape), "received": list(arr.shape)}, status=400)
        pred = np.asarray(model.predict(arr[np.newaxis, :, :], verbose=0)).reshape(-1)
        return JsonResponse({"target": target, "prediction": float(pred[0]), "model_type": "DL"})
    except Exception as exc:
        return JsonResponse({"detail": "DL inference failed.", "error": str(exc)}, status=503)
''', encoding="utf-8")

# Register endpoints without replacing the existing URL configuration.
urls_path = APP_DIR / "urls.py"
urls = urls_path.read_text(encoding="utf-8")
if "from .player_suggestions_day31 import player_suggestions" not in urls:
    urls = "from .player_suggestions_day31 import player_suggestions\n" + urls
if "from .deep_learning_prediction_day31 import deep_learning_prediction" not in urls:
    urls = "from .deep_learning_prediction_day31 import deep_learning_prediction\n" + urls
marker = "urlpatterns = ["
if marker not in urls:
    raise RuntimeError("Could not find urlpatterns in ipl_api/urls.py")
pos = urls.find(marker) + len(marker)
if 'path("players/suggestions/"' not in urls:
    urls = urls[:pos] + '\n    path("players/suggestions/", player_suggestions, name="player-suggestions-day31"),' + urls[pos:]
if 'path("predict/dl/"' not in urls:
    pos = urls.find(marker) + len(marker)
    urls = urls[:pos] + '\n    path("predict/dl/", deep_learning_prediction, name="predict-dl-day31"),' + urls[pos:]
urls_path.write_text(urls, encoding="utf-8")

# Add HTML datalist autocomplete to player-name inputs in dashboard/prediction templates.
template_dir = APP_DIR / "templates" / "ipl_api"
template_changes = []
if template_dir.exists():
    for p in template_dir.rglob("*.html"):
        if not any(k in p.name.lower() for k in ["dashboard", "prediction"]):
            continue
        old = p.read_text(encoding="utf-8", errors="ignore")
        new = old
        def mark(m):
            tag = m.group(0)
            if "data-player-autocomplete" in tag:
                return tag
            if re.search(r'(?:player[_ -]?name|player name)', tag, re.I):
                return tag[:-1] + ' data-player-autocomplete="true" list="ipl-player-suggestions">'
            return tag
        new = re.sub(r'<input\b[^>]*>', mark, new, flags=re.I)
        if "data-player-autocomplete" in new and 'id="ipl-player-suggestions"' not in new:
            new = new.replace("</body>", '<datalist id="ipl-player-suggestions"></datalist>\n</body>') if "</body>" in new else new + '\n<datalist id="ipl-player-suggestions"></datalist>\n'
        if new != old:
            p.write_text(new, encoding="utf-8")
            template_changes.append(str(p.relative_to(ROOT)))

js_dir = APP_DIR / "static" / "ipl_api" / "js"
js_dir.mkdir(parents=True, exist_ok=True)
js_path = js_dir / "player_autocomplete_day31.js"
js_path.write_text('''(() => {
 let timer;
 async function suggest(input) {
  const list = document.getElementById("ipl-player-suggestions");
  if (!list) return;
  const q = input.value.trim();
  if (!q) { list.replaceChildren(); return; }
  try {
   const r = await fetch("/api/players/suggestions/?q=" + encodeURIComponent(q), {headers: {"Accept":"application/json"}});
   if (!r.ok) return;
   const data = await r.json();
   list.replaceChildren(...(data.results || []).map(row => {
    const option = document.createElement("option"); option.value = row.player_name; return option;
   }));
  } catch (_) {}
 }
 document.addEventListener("input", e => {
  if (!e.target.matches("[data-player-autocomplete]")) return;
  clearTimeout(timer); timer = setTimeout(() => suggest(e.target), 180);
 });
})();
''', encoding="utf-8")
script_changes = []
if template_dir.exists():
    for p in template_dir.rglob("*.html"):
        old = p.read_text(encoding="utf-8", errors="ignore")
        if "data-player-autocomplete" not in old or "player_autocomplete_day31.js" in old:
            continue
        tag = '<script src="/static/ipl_api/js/player_autocomplete_day31.js"></script>'
        new = old.replace("</body>", tag + "\n</body>") if "</body>" in old else old + "\n" + tag + "\n"
        p.write_text(new, encoding="utf-8")
        script_changes.append(str(p.relative_to(ROOT)))

for p in [suggestion_path, dl_path, urls_path]:
    compile(p.read_text(encoding="utf-8"), str(p), "exec")
print("Created player autocomplete endpoint and DL sequence endpoint.")
print("Templates updated:", template_changes)
print("Autocomplete script included:", script_changes)
print("Registered GET /api/players/suggestions/ and POST /api/predict/dl/.")
print("DL only accepts preprocessed 5-match sequences; this does not yet convert player names into model-ready sequences.")


Created player autocomplete endpoint and DL sequence endpoint.
Templates updated: ['django_api\\ipl_api\\templates\\ipl_api\\dashboard_day25.html']
Autocomplete script included: ['django_api\\ipl_api\\templates\\ipl_api\\dashboard_day25.html']
Registered GET /api/players/suggestions/ and POST /api/predict/dl/.
DL only accepts preprocessed 5-match sequences; this does not yet convert player names into model-ready sequences.


## 6C. Wire the existing DL prediction panel to the new endpoint

This cell updates known disabled-state wording and adds a browser handler for the DL sequence button. The sample payload is illustrative; use the exact model feature count and saved preprocessing.


In [78]:

from pathlib import Path
import re

template_dir = APP_DIR / "templates" / "ipl_api"
updated_ui = []
for p in template_dir.rglob("*.html") if template_dir.exists() else []:
    old = p.read_text(encoding="utf-8", errors="ignore")
    if not any(x in old for x in ["DL inference is not enabled", "DL endpoint not configured"]):
        continue
    new = old.replace(
        "DL inference is not enabled until a dedicated sequence-model endpoint is registered and validated.",
        "DL sequence endpoint registered. Submit a five-match numeric sequence transformed with the saved Day 12 preprocessing."
    )
    new = new.replace("Unavailable — no verified DL endpoint.", "Ready to validate the saved DL model and sequence input.")
    new = re.sub(r'(<button\b[^>]*?)\sdisabled(\s[^>]*>\s*DL endpoint not configured\s*</button>)',
                 r'\1\2', new, flags=re.I)
    new = new.replace("Provide the exact documented sequence payload only after DL endpoint is implemented.",
                      '{"target":"next_match_runs","sequence":[[0,0],[0,0],[0,0],[0,0],[0,0]],"preprocessed":true}')
    new = new.replace("Exact documented sequence payload required.",
                      '{"target":"next_match_runs","sequence":[[0,0],[0,0],[0,0],[0,0],[0,0]],"preprocessed":true}')
    if new != old:
        p.write_text(new, encoding="utf-8")
        updated_ui.append(str(p.relative_to(ROOT)))

js_dir = APP_DIR / "static" / "ipl_api" / "js"
js_dir.mkdir(parents=True, exist_ok=True)
dl_js = js_dir / "deep_learning_prediction_day31.js"
js_lines = [
'(() => {',
' document.addEventListener("click", async event => {',
'  const button = event.target.closest("button");',
'  if (!button || !/run dl prediction/i.test(button.textContent || "")) return;',
'  let node = button;',
'  for (let i = 0; i < 6 && node && !(node.querySelector && node.querySelector("textarea")); i++) node = node.parentElement;',
'  const area = node && node.querySelector("textarea");',
'  const output = node && node.querySelector("pre, code, .prediction-output, .result");',
'  if (!area) return;',
'  let payload;',
'  try { payload = JSON.parse(area.value); } catch (_) { if (output) output.textContent = "Enter valid JSON."; return; }',
'  button.disabled = true;',
'  try {',
'   const response = await fetch("/api/predict/dl/", {method:"POST", headers:{"Content-Type":"application/json","Accept":"application/json"}, body:JSON.stringify(payload)});',
'   const text = await response.text();',
'   if (output) { try { output.textContent = JSON.stringify(JSON.parse(text), null, 2); } catch (_) { output.textContent = text; } }',
'  } catch (error) { if (output) output.textContent = String(error); }',
'  finally { button.disabled = false; }',
' });',
'})();'
]
dl_js.write_text("\n".join(js_lines)+"\n", encoding="utf-8")
script_tag = '<script src="/static/ipl_api/js/deep_learning_prediction_day31.js"></script>'
script_added = []
for p in template_dir.rglob("*.html") if template_dir.exists() else []:
    old = p.read_text(encoding="utf-8", errors="ignore")
    if "DL sequence endpoint registered" not in old or "deep_learning_prediction_day31.js" in old:
        continue
    new = old.replace("</body>", script_tag + "\n</body>") if "</body>" in old else old + "\n" + script_tag + "\n"
    p.write_text(new, encoding="utf-8")
    script_added.append(str(p.relative_to(ROOT)))
print("DL UI templates changed:", updated_ui)
print("DL handler:", dl_js.relative_to(ROOT))
print("DL script included in:", script_added)
print("The placeholder sequence is illustrative only; model input width and saved preprocessing must match.")


DL UI templates changed: ['django_api\\ipl_api\\templates\\ipl_api\\dashboard_day25.html']
DL handler: django_api\ipl_api\static\ipl_api\js\deep_learning_prediction_day31.js
DL script included in: ['django_api\\ipl_api\\templates\\ipl_api\\dashboard_day25.html']
The placeholder sequence is illustrative only; model input width and saved preprocessing must match.


## 7. Check whether ingestion routes have real implementations

In [79]:
urls = urls_path.read_text(encoding="utf-8")
print("Registered ingestion-related routes:")
for line in urls.splitlines():
    if "ingest" in line.lower() or "health" in line.lower():
        print(line.strip())
print("Potential ingestion files:")
for p in list(APP_DIR.glob("*ingest*.py")) + list((ROOT / "pipeline").glob("*ingest*.py") if (ROOT / "pipeline").exists() else []):
    print("-", p.relative_to(ROOT))
print("Do not report ingestion as successful unless a real ingestion job and persisted run history exist.")


Registered ingestion-related routes:
from .health_views_repair import health_status
path("health/", health_status, name="health-status"),
Potential ingestion files:
Do not report ingestion as successful unless a real ingestion job and persisted run history exist.


## 8. Run Django system check and URL resolver tests

In [80]:
check = subprocess.run([sys.executable, "manage.py", "check"], cwd=DJANGO_DIR, capture_output=True, text=True)
print(check.stdout)
if check.stderr:
    print("STDERR:", check.stderr[-2500:])
print("Django check exit code:", check.returncode)

resolver_code = r'''
from django.urls import resolve
for path in ["/api/dashboard/", "/api/predictions/", "/api/health/", "/api/players/",
             "/api/predict/player/", "/api/eda/opposition/"]:
    try:
        match = resolve(path)
        print(path, "=>", match.url_name, match.func.__module__)
    except Exception as exc:
        print(path, "=> ERROR", type(exc).__name__, str(exc))
'''
res = subprocess.run([sys.executable, "manage.py", "shell", "-c", resolver_code],
                     cwd=DJANGO_DIR, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("Resolver STDERR:", res.stderr[-2500:])
print("Resolver command exit code:", res.returncode)


System check identified no issues (0 silenced).

Django check exit code: 0
12 objects imported automatically (use -v 2 for details).

/api/dashboard/ => ipl-dashboard ipl_api.dashboard_views_day26
/api/predictions/ => ipl-predictions ipl_api.prediction_center_views_day28
/api/health/ => health-status ipl_api.health_views_repair
/api/players/ => player-list ipl_api.views
/api/predict/player/ => predict-player-name-day28 ipl_api.player_name_prediction_views_day28
/api/eda/opposition/ => eda-opposition ipl_api.eda_views

Resolver command exit code: 0


## 9. Generate automatic documentation

In [81]:
report_path = DOC_DIR / "dashboard_live_repair_report.md"
manifest_path = DOC_DIR / "file_manifest.csv"
tracked = [APP_DIR / "urls.py", APP_DIR / "health_views_repair.py", APP_DIR / "views.py", APP_DIR / "eda_views.py", APP_DIR / "player_name_prediction_views_day28.py"]
rows = []
for p in tracked:
    exists = p.exists()
    data = p.read_bytes() if exists else b""
    rows.append({"path": str(p.relative_to(ROOT)), "exists": exists, "bytes": len(data),
                 "sha256": hashlib.sha256(data).hexdigest() if exists else ""})
with manifest_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["path", "exists", "bytes", "sha256"])
    writer.writeheader()
    writer.writerows(rows)
report = [
    "# Dashboard Live Repair Report",
    "",
    "Generated: " + datetime.datetime.now().isoformat(timespec="seconds"),
    "Repository: `" + str(ROOT) + "`",
    "Backup directory: `" + str(BACKUP_DIR.relative_to(ROOT)) + "`",
    "",
    "## Screenshot issues",
    "- Player partial search returns 404.",
    "- ML player-name prediction returns 403.",
    "- Opposition EDA returns 400 due to schema-column mismatch.",
    "- `/api/health/` returns 404.",
    "- Model metrics are present but display/type formatting should be checked.",
    "",
    "## Files",
]
report += [f"- `{r['path']}` — exists: {r['exists']}, bytes: {r['bytes']}" for r in rows]
report += [
    "",
    "## Validation caveat",
    "Review the live outputs of the Django check, URL resolver, and schema audit. A successful system check alone does not prove that prediction, EDA queries, or browser interactions work.",
    "",
    "## Remaining implementation work",
    "- Update player search JS to use the partial-search list endpoint and canonical full name.",
    "- Resolve the ML HTTP 403 from authentication/CSRF settings without disabling CSRF globally.",
    "- Opposition and Recent form are intentionally hidden from the dashboard; do not expose them until implemented and tested.",
    "- Implement ingestion history only if a real ingestion runner and persistence are available.",
]
report_path.write_text("\\n".join(report) + "\\n", encoding="utf-8")
print("Generated report:", report_path)
print("Generated manifest:", manifest_path)


Generated report: D:\Git\Personal\ipl-player-performance\docs\dashboard_live_repair\dashboard_live_repair_report.md
Generated manifest: D:\Git\Personal\ipl-player-performance\docs\dashboard_live_repair\file_manifest.csv


## 10. Acceptance checklist and project status

After running this notebook:
- Restart Django.
- Search `raina`; verify the UI first queries `/api/players/?search=raina` and then uses the canonical full player name.
- Retry ML prediction; confirm the request payload and resolve HTTP 403 based on the view's actual authentication/CSRF settings.
- Run Opposition EDA after using the schema audit output to select a real column.
- Confirm `/api/health/` returns JSON and status 200 while MySQL is reachable.
- Keep ingestion marked unavailable unless a real ingestion runner/history endpoint exists.
- Confirm model comparison shows numeric MAE, RMSE and R² values from saved evaluation results.

The notebook can automate backups, health route registration, schema discovery and diagnostics. The player-search JavaScript, CSRF behavior and opposition query require inspecting the actual source printed above; this notebook will not guess their implementation.


## 6D. Final repair: player autocomplete, CSRF-safe ML request, and actual DL preprocessing

This cell supersedes the earlier experimental Day 31 UI wiring. It keeps changes backed up, makes autocomplete work for player search and ML inputs, sends Django's CSRF token on same-origin POSTs, and makes the DL endpoint load the saved model plus manifest-referenced preprocessing artifacts. It does **not** invent model scores or claim the prediction works until the live checks pass.

In [82]:

# Refresh backup for the exact files this final repair may change.
from pathlib import Path
import shutil, datetime, re, json, os

repair_targets = [
    APP_DIR / "urls.py",
    APP_DIR / "player_suggestions_day31.py",
    APP_DIR / "deep_learning_prediction_day31.py",
    APP_DIR / "dashboard_views_day28.py",
    APP_DIR / "prediction_center_views_day28.py",
]
for folder in [APP_DIR / "templates" / "ipl_api", APP_DIR / "static" / "ipl_api" / "js"]:
    if folder.exists():
        repair_targets += [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in {".html", ".js"}]
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = ROOT / "backups" / f"day31_final_repair_{stamp}"
backup_dir.mkdir(parents=True, exist_ok=True)
for p in sorted(set(repair_targets)):
    if p.exists() and p.is_file():
        dest = backup_dir / p.relative_to(ROOT)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
print("Backup created:", backup_dir)
print("Existing files backed up:", sum(1 for p in set(repair_targets) if p.exists() and p.is_file()))


Backup created: D:\Git\Personal\ipl-player-performance\backups\day31_final_repair_20261024_020141
Existing files backed up: 16


### 6E. Replace player autocomplete JS and make all name fields use it

In [83]:

js_dir = APP_DIR / "static" / "ipl_api" / "js"
js_dir.mkdir(parents=True, exist_ok=True)
autocomplete_js = r"""
(() => {
  const endpoint = "/api/players/suggestions/";
  let timer = null;
  function ensureDatalist() {
    let list = document.getElementById("ipl-player-suggestions");
    if (!list) {
      list = document.createElement("datalist");
      list.id = "ipl-player-suggestions";
      document.body.appendChild(list);
    }
    return list;
  }
  function isPlayerInput(el) {
    if (!(el instanceof HTMLInputElement)) return false;
    const text = [
      el.name, el.id, el.placeholder, el.getAttribute("aria-label"),
      el.closest(".card, section, form")?.innerText?.slice(0, 500)
    ].filter(Boolean).join(" ").toLowerCase();
    return /player.?name|player name|player search|historical performance|batting predictions|bowling predictions/.test(text);
  }
  async function loadSuggestions(input) {
    const query = input.value.trim();
    const list = ensureDatalist();
    if (query.length < 1) { list.replaceChildren(); return; }
    try {
      const response = await fetch(endpoint + "?q=" + encodeURIComponent(query), {
        method: "GET", headers: {"Accept": "application/json"}, credentials: "same-origin"
      });
      if (!response.ok) return;
      const body = await response.json();
      list.replaceChildren(...(body.results || []).map(row => {
        const option = document.createElement("option");
        option.value = row.player_name;
        return option;
      }));
      input.setAttribute("list", list.id);
      input.setAttribute("autocomplete", "off");
    } catch (error) {
      console.warn("Player suggestions unavailable:", error);
    }
  }
  document.addEventListener("focusin", event => {
    if (isPlayerInput(event.target)) {
      event.target.setAttribute("list", "ipl-player-suggestions");
      ensureDatalist();
    }
  });
  document.addEventListener("input", event => {
    if (!isPlayerInput(event.target)) return;
    event.target.setAttribute("list", "ipl-player-suggestions");
    clearTimeout(timer);
    timer = setTimeout(() => loadSuggestions(event.target), 160);
  });
})();
"""
(js_dir / "player_autocomplete_day31.js").write_text(autocomplete_js, encoding="utf-8")
print("Wrote robust autocomplete handler:", js_dir / "player_autocomplete_day31.js")


Wrote robust autocomplete handler: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\js\player_autocomplete_day31.js


### 6F. Fix CSRF for the player-name ML POST without disabling CSRF

In [84]:
# Ensure dashboard sets a CSRF cookie and patch prediction JS with a safe same-origin POST wrapper.
from pathlib import Path
import re

for vp in [APP_DIR / "player_name_prediction_views_day28.py", APP_DIR / "dashboard_views_day28.py"]:
    if not vp.exists():
        continue
    txt = vp.read_text(encoding="utf-8")
    if "def dashboard_day28" in txt and "ensure_csrf_cookie" not in txt:
        txt = "from django.views.decorators.csrf import ensure_csrf_cookie\n" + txt
        txt = re.sub(r"(?m)^def dashboard_day28\(", "@ensure_csrf_cookie\ndef dashboard_day28(", txt, count=1)
        vp.write_text(txt, encoding="utf-8")
        print("CSRF cookie decorator added:", vp.name)

js_files = list((APP_DIR / "static" / "ipl_api" / "js").glob("*.js"))
patched = []
wrapper = r"""
(function day31InstallCsrfFetch() {
  if (window.__day31CsrfFetchInstalled) return;
  window.__day31CsrfFetchInstalled = true;
  const originalFetch = window.fetch.bind(window);
  window.fetch = function(input, init) {
    init = init || {};
    const method = String(init.method || "GET").toUpperCase();
    const url = typeof input === "string" ? input : (input && input.url) || "";
    if (method === "POST" && (url.startsWith("/") || url.startsWith(window.location.origin))) {
      const match = document.cookie.match(/(?:^|; )csrftoken=([^;]+)/);
      const token = match ? decodeURIComponent(match[1]) : "";
      const headers = new Headers(init.headers || (input instanceof Request ? input.headers : undefined));
      if (token && !headers.has("X-CSRFToken")) headers.set("X-CSRFToken", token);
      init = Object.assign({}, init, {headers: headers, credentials: init.credentials || "same-origin"});
    }
    return originalFetch(input, init);
  };
})();
"""
for jp in js_files:
    txt = jp.read_text(encoding="utf-8", errors="ignore")
    if "predict/player" not in txt and "player_name_prediction" not in txt and "predict/batting" not in txt:
        continue
    if "__day31CsrfFetchInstalled" not in txt:
        jp.write_text(wrapper + "\n" + txt, encoding="utf-8")
        patched.append(str(jp.relative_to(ROOT)))
print("CSRF fetch wrapper added to:", patched or "No matching prediction JS found; inspect dashboard JS inventory.")


CSRF fetch wrapper added to: ['django_api\\ipl_api\\static\\ipl_api\\js\\dashboard_day25.js']


### 6G. Replace the DL endpoint with manifest-driven saved preprocessing

In [85]:

# Manifest-driven inference: load model and the imputer/scaler named in Day 12 manifest.
# Accept raw numeric 5 x feature_count rows, apply saved imputer/scaler, then predict.
dl_source = r'''import json
from pathlib import Path
import numpy as np
import pandas as pd
from django.conf import settings
from django.http import JsonResponse
from django.views.decorators.http import require_POST
from django.views.decorators.csrf import csrf_exempt

TARGETS = {"next_match_runs", "next_match_wickets", "next_match_runs_conceded"}

def _path_from_manifest(root, value):
    p = Path(str(value))
    if p.is_absolute() and p.exists():
        return p
    candidates = [root / p, root.parent / p]
    parts = list(p.parts)
    if "models" in parts:
        candidates.append(root.parent.joinpath(*parts[parts.index("models"):]))
        candidates.append(root.joinpath(*parts[parts.index("models"):]))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

@csrf_exempt
@require_POST
def deep_learning_prediction(request):
    try:
        payload = json.loads(request.body.decode("utf-8"))
    except Exception:
        return JsonResponse({"detail": "Body must be valid JSON."}, status=400)
    target = payload.get("target")
    if target not in TARGETS:
        return JsonResponse({"detail": "Unsupported target.", "allowed_targets": sorted(TARGETS)}, status=400)
    try:
        sequence = np.asarray(payload.get("sequence"), dtype=np.float32)
    except (TypeError, ValueError):
        return JsonResponse({"detail": "sequence must contain numeric values."}, status=400)
    if sequence.ndim != 2 or sequence.shape[0] != 5 or sequence.shape[1] < 1:
        return JsonResponse({"detail": "sequence must be 5 x feature_count numeric rows."}, status=400)
    if not np.isfinite(sequence).all():
        return JsonResponse({"detail": "sequence contains NaN or infinite values."}, status=400)
    root = Path(settings.BASE_DIR).parent
    manifest_candidates = [
        root / "reports" / "day12" / "model_artifacts_manifest.csv",
        root / "reports" / "day12_model_artifacts_manifest.csv",
        root / "reports" / "day12" / "model_artifacts_manifest.csv",
    ]
    manifest_path = next((p for p in manifest_candidates if p.exists()), None)
    if manifest_path is None:
        return JsonResponse({"detail": "Day 12 model artifact manifest not found.", "searched": [str(p) for p in manifest_candidates]}, status=503)
    try:
        manifest = pd.read_csv(manifest_path)
        target_col = next((c for c in ["target", "target_name", "prediction_target"] if c in manifest.columns), None)
        if not target_col:
            return JsonResponse({"detail": "Manifest is missing a target column.", "columns": list(manifest.columns)}, status=503)
        row_df = manifest[manifest[target_col].astype(str).str.strip().eq(target)]
        if row_df.empty:
            return JsonResponse({"detail": "No saved DL model for this target.", "target": target}, status=503)
        row = row_df.iloc[0].to_dict()
        model_col = next((c for c in ["model_path", "model_file", "artifact_path"] if c in row and pd.notna(row[c])), None)
        if not model_col:
            return JsonResponse({"detail": "Manifest has no model_path field.", "columns": list(manifest.columns)}, status=503)
        model_path = _path_from_manifest(root, row[model_col])
        if not model_path.exists():
            return JsonResponse({"detail": "Saved model file not found.", "path": str(model_path)}, status=503)
        import tensorflow as tf
        model = tf.keras.models.load_model(model_path)
        feature_width = sequence.shape[1]
        imputer_col = next((c for c in ["imputer_path", "imputer_file"] if c in row and pd.notna(row[c])), None)
        scaler_col = next((c for c in ["scaler_path", "scaler_file"] if c in row and pd.notna(row[c])), None)
        # Saved preprocessing is applied if supplied in the artifact manifest.
        processed = sequence.astype(np.float32)
        import joblib
        if imputer_col:
            imputer_path = _path_from_manifest(root, row[imputer_col])
            if not imputer_path.exists():
                return JsonResponse({"detail": "Saved imputer not found.", "path": str(imputer_path)}, status=503)
            imputer = joblib.load(imputer_path)
            processed = imputer.transform(processed).astype(np.float32)
        if scaler_col:
            scaler_path = _path_from_manifest(root, row[scaler_col])
            if not scaler_path.exists():
                return JsonResponse({"detail": "Saved scaler not found.", "path": str(scaler_path)}, status=503)
            scaler = joblib.load(scaler_path)
            processed = scaler.transform(processed).astype(np.float32)
        expected = tuple(model.input_shape[1:])
        if len(expected) != 2 or expected[0] not in (None, 5) or expected[1] not in (None, processed.shape[1]):
            return JsonResponse({"detail": "Input shape mismatch.", "expected": list(model.input_shape), "received": list(processed.shape)}, status=400)
        pred = np.asarray(model.predict(processed[np.newaxis, :, :], verbose=0)).reshape(-1)
        if pred.size == 0 or not np.isfinite(float(pred[0])):
            return JsonResponse({"detail": "Model returned a non-finite prediction."}, status=503)
        return JsonResponse({"target": target, "prediction": float(pred[0]), "model_type": "DL",
                             "model_file": model_path.name, "sequence_length": 5,
                             "feature_count": int(processed.shape[1]),
                             "preprocessing": {"imputer_loaded": bool(imputer_col), "scaler_loaded": bool(scaler_col)}})
    except Exception as exc:
        return JsonResponse({"detail": "DL inference failed.", "error": str(exc)}, status=503)
'''

dl_path = APP_DIR / "deep_learning_prediction_day31.py"
dl_path.write_text(dl_source, encoding="utf-8")
compile(dl_source, str(dl_path), "exec")
print("Wrote manifest-driven DL endpoint:", dl_path)
print("It applies saved imputer/scaler only when those artifact paths are present in the manifest.")
print("Important: sequence feature order must exactly match Day 12's feature manifest.")


Wrote manifest-driven DL endpoint: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\deep_learning_prediction_day31.py
It applies saved imputer/scaler only when those artifact paths are present in the manifest.
Important: sequence feature order must exactly match Day 12's feature manifest.


### 6H. Enable and wire DL buttons in the actual template

In [86]:

# Use delegated event handling and explicit data attributes, not fragile button-text/DOM guesses.
template_dir = APP_DIR / "templates" / "ipl_api"
changed_templates = []
for p in template_dir.rglob("*.html") if template_dir.exists() else []:
    old = p.read_text(encoding="utf-8", errors="ignore")
    if "DL — Single prediction" not in old and "DL - Single prediction" not in old and "Five-match sequence JSON" not in old:
        continue
    new = old
    # Remove disabled from the DL button specifically, and give it a stable marker.
    new = re.sub(
        r'<button\b(?=[^>]*(?:DL endpoint not configured|Run DL prediction|DL prediction))[^>]*>',
        lambda m: re.sub(r'\sdisabled(?:\s*=\s*(?:"disabled"|\'disabled\'|disabled))?', '', m.group(0), flags=re.I)
                  .replace("<button", '<button data-day31-dl-submit="true"', 1)
        if "data-day31-dl-submit" not in m.group(0) else m.group(0),
        new, flags=re.I
    )
    # If the button was literally labelled as not configured, replace its label.
    new = re.sub(r'(<button\b[^>]*data-day31-dl-submit="true"[^>]*>).*?(</button>)',
                 r'\1Run DL prediction\2', new, flags=re.I | re.S)
    # Update misleading placeholder status, while clearly documenting numeric sequence format.
    new = new.replace("DL endpoint not configured", "Run DL prediction")
    new = new.replace("Submit a five-match numeric sequence transformed with the saved Day 12 preprocessing.",
                      "Enter five chronological match rows in the exact Day 12 feature order; saved imputer/scaler will be applied when present.")
    # Ensure both JS files are loaded.
    for script in ["player_autocomplete_day31.js", "deep_learning_prediction_day31.js"]:
        tag = f'<script src="/static/ipl_api/js/{script}"></script>'
        if script not in new:
            new = new.replace("</body>", tag + "\n</body>") if "</body>" in new else new + "\n" + tag + "\n"
    if new != old:
        p.write_text(new, encoding="utf-8")
        changed_templates.append(str(p.relative_to(ROOT)))
print("Templates updated:", changed_templates or "No matching template found; inspect template inventory in Section 1.")


Templates updated: ['django_api\\ipl_api\\templates\\ipl_api\\dashboard_day25.html']


### 6I. Robust DL form handler

In [87]:

dl_js = r"""
(() => {
  function outputFor(button) {
    const card = button.closest(".card, .prediction-panel, section, form") || button.parentElement;
    return card ? (card.querySelector("pre, code, .prediction-output, .result, .output, [data-result]") || card.lastElementChild) : null;
  }
  document.addEventListener("click", async event => {
    const button = event.target.closest("[data-day31-dl-submit]");
    if (!button) return;
    const card = button.closest(".card, .prediction-panel, section, form") || button.parentElement;
    const area = card && card.querySelector("textarea");
    const output = outputFor(button);
    if (!area) { if (output) output.textContent = "DL input textarea not found in this panel."; return; }
    let payload;
    try { payload = JSON.parse(area.value); }
    catch (_) { if (output) output.textContent = "Enter valid JSON with target and a 5-row sequence."; return; }
    if (!payload.target || !Array.isArray(payload.sequence) || payload.sequence.length !== 5) {
      if (output) output.textContent = 'Expected {"target":"next_match_runs","sequence":[[...],[...],[...],[...],[...]]}.';
      return;
    }
    button.disabled = true;
    if (output) output.textContent = "Running saved DL model…";
    try {
      const response = await fetch("/api/predict/dl/", {
        method: "POST",
        credentials: "same-origin",
        headers: {"Content-Type": "application/json", "Accept": "application/json"},
        body: JSON.stringify(payload)
      });
      const raw = await response.text();
      let body;
      try { body = JSON.parse(raw); } catch (_) { body = {detail: raw.slice(0, 2000)}; }
      if (output) output.textContent = JSON.stringify(body, null, 2);
    } catch (error) {
      if (output) output.textContent = "Request failed: " + String(error);
    } finally {
      button.disabled = false;
    }
  });
})();
"""
(js_dir / "deep_learning_prediction_day31.js").write_text(dl_js, encoding="utf-8")
print("Wrote DL form handler:", js_dir / "deep_learning_prediction_day31.js")


Wrote DL form handler: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\js\deep_learning_prediction_day31.js


### 6J. Compile and check route wiring

In [88]:

for source_path in [
    APP_DIR / "urls.py",
    APP_DIR / "player_suggestions_day31.py",
    APP_DIR / "deep_learning_prediction_day31.py",
]:
    if source_path.exists():
        compile(source_path.read_text(encoding="utf-8"), str(source_path), "exec")
        print("Syntax OK:", source_path.relative_to(ROOT))

check = subprocess.run([sys.executable, "manage.py", "check"], cwd=DJANGO_DIR, capture_output=True, text=True)
print(check.stdout)
if check.stderr:
    print(check.stderr[-2500:])
print("Django check exit code:", check.returncode)


Syntax OK: django_api\ipl_api\urls.py
Syntax OK: django_api\ipl_api\player_suggestions_day31.py
Syntax OK: django_api\ipl_api\deep_learning_prediction_day31.py


System check identified no issues (0 silenced).

Django check exit code: 0


## Required live acceptance tests

After running all cells, restart the Django server and test:

1. Type `raina` in Player Search. Suggestions should appear from `/api/players/suggestions/?q=raina`; choose a full canonical name such as `SK Raina`.
2. Inspect DevTools → Network for `/api/predict/player/`. The response must be JSON, not Django's CSRF HTML page. If still 403, check whether the browser received `csrftoken` and whether the actual ML JS includes the `X-CSRFToken` header.
3. For DL, use the exact feature count/order from the Day 12 feature manifest. Do not use the illustrative all-zero 2-feature sequence unless the saved model really expects two features. The endpoint should return a real prediction or a specific schema/artifact error.
4. Run `/api/health/` and confirm HTTP 200.
5. If a DL button remains disabled, the template did not match the expected markup; inspect the generated template changes list before manually editing.

This notebook can wire the API and UI, but only the live response confirms model inference works. The saved feature manifest is authoritative for sequence order and width.

## 6K. Final targeted fix: guaranteed CSRF bootstrap and DL inference request wiring

This section adds a same-origin CSRF bootstrap endpoint, makes prediction requests fetch the token before POSTing, and rewrites the DL handler so the response is displayed in the correct prediction panel. It does not disable Django CSRF middleware. The DL model endpoint still returns a clear 4xx/5xx message if its Day 12 manifest/artifacts are missing or incompatible.

In [89]:

# Back up and add a CSRF bootstrap endpoint; then register it safely.
from pathlib import Path
import re, shutil, datetime, json, subprocess, sys

repair_now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
for p in [APP_DIR / "urls.py", APP_DIR / "deep_learning_prediction_day31.py"]:
    if p.exists():
        backup = p.with_name(p.name + f".day31_final_{repair_now}.bak")
        shutil.copy2(p, backup)
        print("Backup:", backup.name)

csrf_path = APP_DIR / "csrf_bootstrap_day31.py"
csrf_path.write_text("""from django.http import JsonResponse
from django.middleware.csrf import get_token
from django.views.decorators.csrf import ensure_csrf_cookie
from django.views.decorators.http import require_GET

@require_GET
@ensure_csrf_cookie
def csrf_bootstrap(request):
    return JsonResponse({"csrfToken": get_token(request)})
""", encoding="utf-8")

urls_path = APP_DIR / "urls.py"
urls_text = urls_path.read_text(encoding="utf-8")
if "csrf_bootstrap_day31" not in urls_text:
    import_line = "from .csrf_bootstrap_day31 import csrf_bootstrap"
    # insert after imports
    lines = urls_text.splitlines()
    last_import = max((i for i,line in enumerate(lines) if line.startswith("import ") or line.startswith("from ")), default=-1)
    lines.insert(last_import + 1, import_line)
    urls_text = "\n".join(lines) + "\n"
if 'path("csrf/", csrf_bootstrap' not in urls_text and "path('csrf/', csrf_bootstrap" not in urls_text:
    # Insert before closing urlpatterns list bracket.
    m = re.search(r"(?s)(urlpatterns\s*=\s*\[)(.*?)(\])", urls_text)
    if not m:
        raise RuntimeError("Could not safely find urlpatterns list in ipl_api/urls.py")
    body = m.group(2)
    body += '\n    path("csrf/", csrf_bootstrap, name="csrf-bootstrap-day31"),\n'
    urls_text = urls_text[:m.start()] + m.group(1) + body + m.group(3) + urls_text[m.end():]
urls_path.write_text(urls_text, encoding="utf-8")
print("Added CSRF bootstrap route: /api/csrf/")

# Rewrite the fetch wrapper so it gets a cookie/token before every same-origin POST.
js_dir = APP_DIR / "static" / "ipl_api" / "js"
js_dir.mkdir(parents=True, exist_ok=True)
csrf_js = r"""(() => {
  if (window.__day31GuaranteedCsrf) return;
  window.__day31GuaranteedCsrf = true;
  const originalFetch = window.fetch.bind(window);
  async function tokenFromServer() {
    const r = await originalFetch("/api/csrf/", {
      method: "GET", credentials: "same-origin", headers: {"Accept": "application/json"}
    });
    if (!r.ok) throw new Error("CSRF bootstrap failed (HTTP " + r.status + ")");
    const data = await r.json();
    return data.csrfToken || "";
  }
  window.fetch = async function(input, init) {
    init = init || {};
    const method = String(init.method || (input instanceof Request ? input.method : "GET")).toUpperCase();
    const url = typeof input === "string" ? input : (input && input.url) || "";
    const sameOrigin = url.startsWith("/") || url.startsWith(window.location.origin);
    if (sameOrigin && ["POST", "PUT", "PATCH", "DELETE"].includes(method)) {
      const token = await tokenFromServer();
      const headers = new Headers(init.headers || (input instanceof Request ? input.headers : undefined));
      if (!headers.has("X-CSRFToken")) headers.set("X-CSRFToken", token);
      init = Object.assign({}, init, {headers, credentials: "same-origin"});
    }
    return originalFetch(input, init);
  };
})();
"""
(js_dir / "day31_csrf_bootstrap.js").write_text(csrf_js, encoding="utf-8")

# Rewrite DL handler with stable panel lookup and informative server response.
dl_js = r"""(() => {
  function panelFor(button) {
    return button.closest(".card") || button.closest(".col") || button.parentElement.parentElement;
  }
  function resultBox(panel) {
    return panel.querySelector("[data-day31-dl-result]") ||
      panel.querySelector("pre") || panel.querySelector("code") ||
      panel.querySelector(".result, .output, .prediction-output") ||
      panel.lastElementChild;
  }
  document.addEventListener("click", async event => {
    const button = event.target.closest("[data-day31-dl-submit]");
    if (!button) return;
    const panel = panelFor(button);
    const area = panel.querySelector("textarea");
    const out = resultBox(panel);
    const show = value => { if (out) out.textContent = typeof value === "string" ? value : JSON.stringify(value, null, 2); };
    if (!area) { show("DL input field not found in this panel."); return; }
    let payload;
    try { payload = JSON.parse(area.value); }
    catch (e) { show("Invalid JSON: include target and a 5-row sequence."); return; }
    if (!["next_match_runs", "next_match_wickets", "next_match_runs_conceded"].includes(payload.target) ||
        !Array.isArray(payload.sequence) || payload.sequence.length !== 5 ||
        !payload.sequence.every(row => Array.isArray(row) && row.length > 0 && row.every(v => typeof v === "number" && Number.isFinite(v)))) {
      show({detail: "Payload must include a supported target and five chronological numeric rows.", example:
        {target:"next_match_runs", sequence:[[0,0],[0,0],[0,0],[0,0],[0,0]]}});
      return;
    }
    button.disabled = true;
    show("Loading saved Day 12 model and preprocessing…");
    try {
      const response = await fetch("/api/predict/dl/", {
        method: "POST", credentials: "same-origin",
        headers: {"Content-Type":"application/json", "Accept":"application/json"},
        body: JSON.stringify(payload)
      });
      const raw = await response.text();
      let data; try { data = JSON.parse(raw); } catch (_) { data = {detail: raw.slice(0, 1800)}; }
      show({http_status: response.status, ...data});
    } catch (err) { show("Request failed: " + String(err)); }
    finally { button.disabled = false; }
  });
})();
"""
(js_dir / "deep_learning_prediction_day31.js").write_text(dl_js, encoding="utf-8")

# Ensure both scripts are included in dashboard templates that contain prediction cards.
templates = APP_DIR / "templates" / "ipl_api"
changed = []
for tp in templates.rglob("*.html") if templates.exists() else []:
    old = tp.read_text(encoding="utf-8", errors="ignore")
    if not any(x in old for x in ["ML — Single prediction", "Batting Predictions", "Bowling Predictions", "data-day31-dl-submit"]):
        continue
    new = old
    for script in ["day31_csrf_bootstrap.js", "deep_learning_prediction_day31.js"]:
        tag = f'<script src="/static/ipl_api/js/{script}"></script>'
        if script not in new:
            if "</body>" in new: new = new.replace("</body>", tag + "\n</body>")
            else: new += "\n" + tag + "\n"
    if new != old:
        tp.write_text(new, encoding="utf-8")
        changed.append(str(tp.relative_to(ROOT)))
print("Templates with script tags added:", changed)

print("CSRF JS and DL handler written.")


Backup: urls.py.day31_final_20261024_020145.bak
Backup: deep_learning_prediction_day31.py.day31_final_20261024_020145.bak
Added CSRF bootstrap route: /api/csrf/
Templates with script tags added: ['django_api\\ipl_api\\templates\\ipl_api\\dashboard_day25.html']
CSRF JS and DL handler written.


## 6L. Verify route, Python syntax, and Django configuration

A passing `manage.py check` confirms configuration/syntax only; it does not prove the trained DL artifacts can predict. After running this cell, restart Django and test the browser. The DL response will state whether a manifest, model, imputer/scaler, or feature shape is incompatible.

In [90]:

for p in [APP_DIR / "urls.py", APP_DIR / "csrf_bootstrap_day31.py", APP_DIR / "deep_learning_prediction_day31.py"]:
    if p.exists():
        compile(p.read_text(encoding="utf-8"), str(p), "exec")
        print("Syntax OK:", p.relative_to(ROOT))
check = subprocess.run([sys.executable, "manage.py", "check"], cwd=DJANGO_DIR, capture_output=True, text=True)
print(check.stdout)
if check.stderr: print(check.stderr[-3000:])
print("manage.py check exit code:", check.returncode)


Syntax OK: django_api\ipl_api\urls.py
Syntax OK: django_api\ipl_api\csrf_bootstrap_day31.py
Syntax OK: django_api\ipl_api\deep_learning_prediction_day31.py
System check identified no issues (0 silenced).

manage.py check exit code: 0
